# 03 · Fast models: Ridge, PCA + Ridge, Decision Tree, LightGBM, XGBoost, CatBoost
Same protocol for all (see `src/models.py`): tune on a 300k-row month-stratified subset of **train**, early-stop and
score on **validation**, refit the best configuration on all train rows, save validation predictions.
Results are saved after every model, so you can stop and restart: finished models are skipped.
Rough time on a laptop: Ridge/PCA/Tree seconds-minutes, LightGBM minutes, XGBoost and CatBoost 10-40 minutes each.

In [1]:
import sys, os, json, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
from src import evaluate as ev, metrics as M, models as mo
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10, "axes.titleweight": "bold"})
def savefig(fig, name):
    fig.tight_layout(); fig.savefig(ev.FIGS / f"{name}.png", dpi=150); plt.show()
print("results folder:", ev.RESULTS, "| smoke mode:", mo.SMOKE, "| threads:", mo.N_JOBS)

results folder: C:\dev\traffic-prediction\results | smoke mode: False | threads: 11


## Run the models (both horizons)

In [ ]:
FAST = ["ridge", "pca_ridge", "dt", "lgbm", "xgb", "cat"]       # ridge first: PCA reuses its best penalty
summary = []
for name in FAST:
    for h in ev.HORIZONS:
        r = mo.tune_model(name, h)
        summary.append({"model": mo.LABEL[name], "horizon_min": h, "val MAE": r["val"]["MAE"], "RMSE": r["val"]["RMSE"],
                        "WAPE_%": r["val"]["WAPE"] * 100, "best_iter": r["best_iter"], "fit_s": r["fit_s_train"], "params": r["params"]})
display(pd.DataFrame(summary).round(2))

[ridge h=15] cfg 1/6 {'alpha': 0.01} -> val MAE 57.39  (1s)
[ridge h=15] cfg 2/6 {'alpha': 0.1} -> val MAE 57.39  (1s)
[ridge h=15] cfg 3/6 {'alpha': 1} -> val MAE 57.39  (0s)
[ridge h=15] cfg 4/6 {'alpha': 10} -> val MAE 57.39  (0s)
[ridge h=15] cfg 5/6 {'alpha': 100} -> val MAE 57.42  (0s)
[ridge h=15] cfg 6/6 {'alpha': 1000} -> val MAE 57.77  (0s)
[ridge h=15] BEST {'alpha': 0.01} best_iter=None  val MAE 57.36  (full fit 3s, total 0.1 min)
[ridge h=60] cfg 1/6 {'alpha': 0.01} -> val MAE 80.52  (1s)
[ridge h=60] cfg 2/6 {'alpha': 0.1} -> val MAE 80.52  (1s)
[ridge h=60] cfg 3/6 {'alpha': 1} -> val MAE 80.52  (0s)
[ridge h=60] cfg 4/6 {'alpha': 10} -> val MAE 80.52  (0s)
[ridge h=60] cfg 5/6 {'alpha': 100} -> val MAE 80.52  (0s)
[ridge h=60] cfg 6/6 {'alpha': 1000} -> val MAE 80.64  (0s)
[ridge h=60] BEST {'alpha': 0.01} best_iter=None  val MAE 80.49  (full fit 3s, total 0.1 min)
[pca_ridge h=15] cfg 1/5 {'k': 5, 'alpha': 0.01} -> val MAE 88.78  (0s)
[pca_ridge h=15] cfg 2/5 {'k': 10,

## PCA: how many components are needed?
If PCA with k components matches plain Ridge, the 31 features are redundant (as the correlation heatmap suggested).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=False)
for ax, h in zip(axes, ev.HORIZONS):
    lg = pd.read_csv(ev.LOGS / f"tuning_pca_ridge_h{h}.csv").sort_values("k")
    ax.plot(lg["k"], lg["val_MAE"], "o-", color="#0072B2", label="PCA + Ridge")
    rg = pd.read_csv(ev.LOGS / f"tuning_ridge_h{h}.csv")["val_MAE"].min()
    ax.axhline(rg, color="#D55E00", ls="--", label="plain Ridge")
    ax.set_xlabel("number of components k"); ax.set_ylabel("validation MAE"); ax.set_title(f"{h}-minute horizon"); ax.legend()
savefig(fig, "model_pca_curve")

## Decision tree: depth vs error (over- vs under-fitting)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, h in zip(axes, ev.HORIZONS):
    lg = pd.read_csv(ev.LOGS / f"tuning_dt_h{h}.csv")
    for leaf, g in lg.groupby("min_samples_leaf"):
        g = g.sort_values("max_depth"); ax.plot(g["max_depth"], g["val_MAE"], "o-", label=f"min leaf {leaf}")
    ax.set_xlabel("max depth"); ax.set_ylabel("validation MAE"); ax.set_title(f"{h}-minute horizon"); ax.legend()
savefig(fig, "model_tree_depth")

## Boosting: tuning trials and how many trees were kept

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for ax, name in zip(axes, ["lgbm", "xgb", "cat"]):
    for h, col in zip(ev.HORIZONS, ["#0072B2", "#D55E00"]):
        lg = pd.read_csv(ev.LOGS / f"tuning_{name}_h{h}.csv")
        ax.scatter(lg["best_iter"], lg["val_MAE"], color=col, label=f"{h} min", s=30)
    ax.set_title(mo.LABEL[name]); ax.set_xlabel("trees kept by early stopping"); ax.set_ylabel("validation MAE"); ax.legend()
savefig(fig, "model_boosting_trials")

## Slow models: status
Random Forest, AdaBoost and Gradient Boosting run as scripts (they can take hours). In a **terminal** (venv active):
```
python scripts/run_model.py tune --models rf --h 15 60
python scripts/run_model.py tune --models ada gb --h 15 60
```
This cell shows what is finished.

In [ ]:
for name in ["rf", "ada", "gb"]:
    for h in ev.HORIZONS:
        r = ev.read_best(name, h)
        print(f"{mo.LABEL[name]:18s} h={h:2d}  " + (f"done  val MAE {r['val']['MAE']:.2f}  params {r['params']}" if r else "not run yet"))